# Function Calling with Gemma 3

Adopted from 
- https://github.com/arjunprabhulal/function-calling-gemma3
- https://medium.com/@karust/now-you-can-search-on-google-for-free-solution-with-api-7699954325b1

In [1]:
import ollama
MODEL = "gemma3:12b"

In [4]:
from prompts import SYSTEM_PROMPT

In [5]:
def google_search(search_parameters: dict) -> str:
    # FAKE API 
    return "The president of the U.S elected in 2025 was Donald Trump."

In [6]:
from pydantic import BaseModel, Field
from typing import Optional, Dict, Any, List
import json

class FunctionCall(BaseModel):
    """Model for function calls from the LLM"""
    name: str = Field(..., description="Name of the function to call")
    parameters: Dict[str, Any] = Field(..., description="Parameters for the function")

def parse_function_call(response: str) -> Optional[FunctionCall]:
    """Parse the model's response to extract function calls"""
    try:
        # Clean the response and find JSON structure
        response = response.strip()
        start_idx = response.find('{')
        end_idx = response.rfind('}') + 1
        
        if start_idx == -1 or end_idx == 0:
            return None
            
        json_str = response[start_idx:end_idx]
        data = json.loads(json_str)
        return FunctionCall(**data)
    except Exception as e:
        print(f"Error parsing function call: {str(e)}")
        return None

In [7]:
def chat(user_prompt: str) -> str:
    response = ollama.chat(model=MODEL,
                           messages=[{"role": "system", "content": SYSTEM_PROMPT},
                                     {"role": "user", "content": user_prompt}]
                          )
    model_response = response['message']['content']
    print(model_response)
    
    function_call = parse_function_call(model_response)
    print(function_call)

    if function_call:
        result = ""
        if function_call.name == "google_search":
            result = google_search(function_call.parameters)
        else:
            pass
        ALTERNATIVE_PROMPT = f"""
        You are an intelligent assistent. Answering to the user question using the following information:
        {result}
        """
        response = ollama.chat(model=MODEL,
                       messages=[{"role": "system", "content": ALTERNATIVE_PROMPT},
                                 {"role": "user", "content": user_prompt}]
                      )
        model_response = response['message']['content']
        print(model_response)            
    return model_response

In [8]:
prompt = "Who was elected as president in the U.S. in 2025?"
reponse = chat(prompt)

{
    "name": "google_search",
    "parameters": {
        "query": "who was elected president of the US in 2025"
    }
}
name='google_search' parameters={'query': 'who was elected president of the US in 2025'}
According to the information I have, Donald Trump was elected as president of the U.S. in 2025.
